##### Noisy QSVM (SVC Precomputed Kernel) - Spambase

In [2]:
%pip install qiskit qiskit-machine-learning qiskit-aer seaborn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 12.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 96.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 85.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 53.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 90.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 76.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 102.5 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [3]:
import qiskit, qiskit_aer, qiskit_machine_learning
print("Qiskit:", qiskit.__version__)
print("Aer:", qiskit_aer.__version__)
print("QML:", qiskit_machine_learning.__version__)

Qiskit: 2.2.3
Aer: 0.17.2
QML: 0.9.0


In [4]:
# --- Import Libraries ---
import pandas as pd
import numpy as np
import time
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif, VarianceThreshold
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score, recall_score, balanced_accuracy_score

In [5]:
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

In [6]:
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

In [7]:
# --- Qiskit Imports ---
from qiskit.circuit.library import ZZFeatureMap
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error, ReadoutError
from qiskit.primitives import StatevectorSampler as Sampler
from qiskit_aer.primitives import SamplerV2 as AerSampler
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_machine_learning.state_fidelities import ComputeUncompute
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.algorithms import QSVC, PegasosQSVC

##### Load Dataset

In [8]:
# --- Import Spambase Column Names ---
spambase_columns = [
    "word_freq_make",
    "word_freq_address",
    "word_freq_all",
    "word_freq_3d",
    "word_freq_our",
    "word_freq_over",
    "word_freq_remove",
    "word_freq_internet",
    "word_freq_order",
    "word_freq_mail",
    "word_freq_receive",
    "word_freq_will",
    "word_freq_people",
    "word_freq_report",
    "word_freq_addresses",
    "word_freq_free",
    "word_freq_business",
    "word_freq_email",
    "word_freq_you",
    "word_freq_credit",
    "word_freq_your",
    "word_freq_font",
    "word_freq_000",
    "word_freq_money",
    "word_freq_hp",
    "word_freq_hpl",
    "word_freq_george",
    "word_freq_650",
    "word_freq_lab",
    "word_freq_labs",
    "word_freq_telnet",
    "word_freq_857",
    "word_freq_data",
    "word_freq_415",
    "word_freq_85",
    "word_freq_technology",
    "word_freq_1999",
    "word_freq_parts",
    "word_freq_pm",
    "word_freq_direct",
    "word_freq_cs",
    "word_freq_meeting",
    "word_freq_original",
    "word_freq_project",
    "word_freq_re",
    "word_freq_edu",
    "word_freq_table",
    "word_freq_conference",
    "char_freq_;",
    "char_freq_(",
    "char_freq_[",
    "char_freq_!",
    "char_freq_$",
    "char_freq_#",
    "capital_run_length_average",
    "capital_run_length_longest",
    "capital_run_length_total",
    "label"
]

# --- 1. Load the Spambase Dataset (LOCAL PATH) ---
# file_path = '/kaggle/input/spambase/spambase.data'
# file_path = r'C:\Users\User\Documents\MyProjects\FYP_ResearchProject\data\spambase\spambase.data'
file_path = "/home/azureuser/cloudfiles/code/data/spambase/spambase.data"
df = pd.read_csv(file_path, header=None, names=spambase_columns)
df.drop_duplicates(inplace=True)

print(f"Dataset loaded: {df.shape[0]} samples, {df.shape[1]} features")

Dataset loaded: 4210 samples, 58 features


##### Noise Model Implementation and Error Mitigation Functions

In [9]:
# Base error rates (realistic NISQ device)
NOISE_CONFIGS = {
    'low': {'p_1q': 0.0001, 'p_2q': 0.001, 'p_ro': 0.002},
    'standard': {'p_1q': 0.001,  'p_2q': 0.01,  'p_ro': 0.02},
    'high': {'p_1q': 0.005,  'p_2q': 0.05,  'p_ro': 0.10}
}

def get_scaled_noise_model(scale_factor=1.0, level='standard'):
    """
    Build a noise model with scaled error probabilities for ZNE.
    Uses formula: p_scaled = 1 - (1-p)^scale_factor
    """
    config = NOISE_CONFIGS.get(level, NOISE_CONFIGS['standard'])
    
    p_1q = config['p_1q']
    p_2q = config['p_2q']
    p_ro = config['p_ro']

    p_1q_scaled = 1 - (1 - p_1q)**scale_factor
    p_2q_scaled = 1 - (1 - p_2q)**scale_factor
    p_ro_scaled = 1 - (1 - p_ro)**scale_factor
    
    noise_model = NoiseModel()
    noise_model.add_all_qubit_quantum_error(depolarizing_error(p_1q_scaled, 1), ['u1', 'u2', 'u3'])
    noise_model.add_all_qubit_quantum_error(depolarizing_error(p_2q_scaled, 2), ['cx'])
    readout_error = ReadoutError([[1 - p_ro_scaled, p_ro_scaled], [p_ro_scaled, 1 - p_ro_scaled]])
    noise_model.add_all_qubit_readout_error(readout_error)
    
    backend = AerSimulator(noise_model=noise_model, seed_simulator=12345)
    pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
    
    return noise_model, backend, pm, config

print("Noise model factory function ready!")

Noise model factory function ready!


In [10]:
# ==========================================
# READOUT ERROR MITIGATION (REM) FUNCTION
# ==========================================

def apply_rem_to_kernel(kernel_matrix, p_ro, n_qubits):
    """
    Apply Readout Error Mitigation to a kernel matrix.
    
    The correction formula for fidelity with symmetric readout error:
    K_corrected = (K_noisy - bias) / correction_factor
    """
    correction_factor = (1 - 2 * p_ro) ** n_qubits
    
    if abs(correction_factor) < 1e-10:
        print("Warning: Correction factor too small, using raw kernel")
        return kernel_matrix
    
    bias = 0.5 * (1 - correction_factor)
    corrected_kernel = (kernel_matrix - bias) / correction_factor
    
    # Clip to valid kernel range [0, 1]
    corrected_kernel = np.clip(corrected_kernel, 0, 1)
    
    # Ensure diagonal is exactly 1 (self-similarity)
    if kernel_matrix.shape[0] == kernel_matrix.shape[1]:
        np.fill_diagonal(corrected_kernel, 1.0)
    
    return corrected_kernel

print("REM function ready!")

REM function ready!


##### Experiment Configurations

In [11]:
# ==========================================
# EXPERIMENT CONFIGURATIONS (ZNE+REM COMBINED)
# Fixed Configuration from Classical SVM Best Result
# ==========================================
#
# Base Configuration (from Classical RBF Best):
# - samples: 300
# - k_features: 3
# - C: 1.0 (fixed, no grid search)
#

experiments = [
    # --- BASELINE (Matches Classical Best) ---
    {'id': 'Baseline', 'samples': 300, 'k_features': 3, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard'},
    
    # --- EXP 1: Shot Noise Effect ---
    {'id': 'Exp1_128shots',  'samples': 300, 'k_features': 3, 'shots': 128,  'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard'},
    {'id': 'Exp1_512shots',  'samples': 300, 'k_features': 3, 'shots': 512,  'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard'},


    # --- EXP 2: Circuit Depth (Reps) Effect ---
    # Baseline serves as Exp2_Reps1
    {'id': 'Exp2_Reps2', 'samples': 300, 'k_features': 3, 'shots': 1024, 'reps': 2, 'entanglement': 'linear', 'noise_level': 'standard'},


    # --- EXP 3: Entanglement Topology ---
    # Baseline serves as Exp3_Linear
    {'id': 'Exp3_Circular', 'samples': 300, 'k_features': 3, 'shots': 1024, 'reps': 1, 'entanglement': 'circular', 'noise_level': 'standard'},
    {'id': 'Exp3_Full',     'samples': 300, 'k_features': 3, 'shots': 1024, 'reps': 1, 'entanglement': 'full',     'noise_level': 'standard'},


    # --- EXP 4: Noise Level Effect ---
    {'id': 'Exp4_LowNoise',  'samples': 300, 'k_features': 3, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'low'},
    # Baseline serves as Exp4_StdNoise
    {'id': 'Exp4_HighNoise', 'samples': 300, 'k_features': 3, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'high'},
]

print(f"Total experiments configured: {len(experiments)}")
print("Running might take hours or even several days")


Total experiments configured: 8
Running might take hours or even several days


In [12]:
# ===================================================================
# QUICK CHECK: Verify Cached Kernel Files
# ===================================================================
import os
import numpy as np

kernel_dir = 'kernels_em_znerem_spambase_final'

# List all experiments
experiment_ids = ['Baseline', 'Exp1_128shots', 'Exp1_512shots', 'Exp2_Reps2', 
                  'Exp3_Circular', 'Exp3_Full', 'Exp4_LowNoise', 'Exp4_HighNoise']

print("=" * 80)
print("CACHED KERNEL FILE VERIFICATION")
print("=" * 80)

for exp_id in experiment_ids:
    print(f"\n📂 {exp_id}:")
    
    # Check if all 4 files exist
    files = [
        f'kernel_train_scale1_{exp_id}.npy',
        f'kernel_test_scale1_{exp_id}.npy',
        f'kernel_train_scale3_{exp_id}.npy',
        f'kernel_test_scale3_{exp_id}.npy'
    ]
    
    all_exist = True
    for fname in files:
        fpath = f'{kernel_dir}/{fname}'
        if os.path.exists(fpath):
            # Load and check shape
            arr = np.load(fpath)
            print(f"   ✓ {fname:40s} Shape: {arr.shape}")
        else:
            print(f"   ✗ {fname:40s} MISSING!")
            all_exist = False
    
    if all_exist:
        print(f"   ✅ All kernels cached for {exp_id}")
    else:
        print(f"   ⚠️  Incomplete cache for {exp_id}")

print("\n" + "=" * 80)


CACHED KERNEL FILE VERIFICATION

📂 Baseline:
   ✓ kernel_train_scale1_Baseline.npy         Shape: (300, 300)
   ✓ kernel_test_scale1_Baseline.npy          Shape: (129, 300)
   ✓ kernel_train_scale3_Baseline.npy         Shape: (300, 300)


   ✓ kernel_test_scale3_Baseline.npy          Shape: (129, 300)
   ✅ All kernels cached for Baseline

📂 Exp1_128shots:
   ✗ kernel_train_scale1_Exp1_128shots.npy    MISSING!
   ✗ kernel_test_scale1_Exp1_128shots.npy     MISSING!
   ✗ kernel_train_scale3_Exp1_128shots.npy    MISSING!
   ✗ kernel_test_scale3_Exp1_128shots.npy     MISSING!
   ⚠️  Incomplete cache for Exp1_128shots

📂 Exp1_512shots:
   ✗ kernel_train_scale1_Exp1_512shots.npy    MISSING!
   ✗ kernel_test_scale1_Exp1_512shots.npy     MISSING!
   ✗ kernel_train_scale3_Exp1_512shots.npy    MISSING!
   ✗ kernel_test_scale3_Exp1_512shots.npy     MISSING!
   ⚠️  Incomplete cache for Exp1_512shots

📂 Exp2_Reps2:
   ✗ kernel_train_scale1_Exp2_Reps2.npy       MISSING!
   ✗ kernel_test_scale1_Exp2_Reps2.npy        MISSING!
   ✗ kernel_train_scale3_Exp2_Reps2.npy       MISSING!
   ✗ kernel_test_scale3_Exp2_Reps2.npy        MISSING!
   ⚠️  Incomplete cache for Exp2_Reps2

📂 Exp3_Circular:
   ✗ kernel_train_scale1_Exp3_Circular.npy    M

##### Main Experiment

In [ ]:
# ===================================================================
# MAIN EXPERIMENT LOOP - ERROR MITIGATED QSVM (ZNE + REM COMBINED)
# ===================================================================
import os
import time
from sklearn.feature_selection import VarianceThreshold

# Setup kernel directory
kernel_dir = 'kernels_em_znerem_spambase_final'
os.makedirs(kernel_dir, exist_ok=True)

# ZNE scales for Richardson extrapolation
ZNE_SCALES = [1.0, 3.0]

# Store all results
all_results = []

# Split features and target once
X = df.drop('label', axis=1)
y = df['label']

for exp_num, config in enumerate(experiments, 1):
    print("\n" + "=" * 80)
    print(f"EXPERIMENT {exp_num}/{len(experiments)}: {config['id']}")
    print("=" * 80)
    print(f"  Samples: {config['samples']}")
    print(f"  K Features: {config['k_features']}")
    print(f"  Shots: {config['shots']}")
    print(f"  Reps: {config['reps']}")
    print(f"  Entanglement: {config['entanglement']}")
    print(f"  Noise Level: {config.get('noise_level', 'standard')}")
    print("=" * 80)
    
    # Check if kernel files exist for caching
    train_file_scale1 = f'{kernel_dir}/kernel_train_scale1_{config["id"]}.npy'
    test_file_scale1 = f'{kernel_dir}/kernel_test_scale1_{config["id"]}.npy'
    train_file_scale3 = f'{kernel_dir}/kernel_train_scale3_{config["id"]}.npy'
    test_file_scale3 = f'{kernel_dir}/kernel_test_scale3_{config["id"]}.npy'
    
    skip_quantum = (os.path.exists(train_file_scale1) and 
                    os.path.exists(test_file_scale1) and
                    os.path.exists(train_file_scale3) and 
                    os.path.exists(test_file_scale3))
    
    # ===================================================================
    # 1. CREATE DATASET BASED ON SAMPLE SIZE
    # ===================================================================
    
    train_samples = config['samples']
    subset_size = int(round(train_samples / 0.7))
    
    # First sample subset from full dataset
    X_subset, _, y_subset, _ = train_test_split(
        X, y,
        train_size=subset_size,
        stratify=y,
        random_state=42
    )
    
    # Then do 70:30 split
    X_train, X_test, y_train, y_test = train_test_split(
        X_subset, y_subset,
        test_size=0.30,
        random_state=42,
        stratify=y_subset
    )
    
    print(f"\nDataset created: {X_train.shape[0]} train, {X_test.shape[0]} test")
    
    # ===================================================================
    # 2. SCALING
    # ===================================================================
    
    # Variance filtering & scaling
    selector_variance = VarianceThreshold(threshold=0)
    X_train_filtered = selector_variance.fit_transform(X_train)
    X_test_filtered = selector_variance.transform(X_test)
    remaining_cols = X_train.columns[selector_variance.get_support()]
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_filtered)
    X_test_scaled = scaler.transform(X_test_filtered)
    X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=remaining_cols)
    X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=remaining_cols)
    
    print("Data scaled successfully")
    
    # ===================================================================
    # 3. CORRELATION-BASED FEATURE DROPPING
    # ===================================================================
    
    THRESH = 0.9
    corr_matrix_train = X_train_scaled_df.corr().abs()
    upper_triangle = corr_matrix_train.where(
        np.triu(np.ones(corr_matrix_train.shape), k=1).astype(bool)
    )
    
    columns_to_drop = set()
    for column in upper_triangle.columns:
        high_corr_partners = upper_triangle.index[upper_triangle[column] > THRESH].tolist()
        if high_corr_partners:
            for partner in high_corr_partners:
                corr_main_vs_target = y_train.corr(X_train_scaled_df[column])
                corr_partner_vs_target = y_train.corr(X_train_scaled_df[partner])
                
                if abs(corr_main_vs_target) < abs(corr_partner_vs_target):
                    columns_to_drop.add(column)
                else:
                    columns_to_drop.add(partner)
    
    to_drop_final = sorted(list(columns_to_drop))
    X_train_selected = X_train_scaled_df.drop(columns=to_drop_final)
    X_test_selected = X_test_scaled_df.drop(columns=to_drop_final)
    
    print(f"Dropped {len(to_drop_final)} highly correlated features")
    
    # ===================================================================
    # 4. SELECTKBEST
    # ===================================================================
    
    k_features = config['k_features']
    selector = SelectKBest(score_func=mutual_info_classif, k=k_features)
    X_train_kbest = selector.fit_transform(X_train_selected, y_train)
    X_test_kbest = selector.transform(X_test_selected)
    
    selected_features = X_train_selected.columns[selector.get_support()].tolist()
    print(f"SelectKBest: Selected {k_features} features:")
    for i, feat in enumerate(selected_features, 1):
        print(f"     {i}. {feat}")
    
    # ===================================================================
    # 5. COMPUTE OR LOAD QUANTUM KERNELS (ZNE SCALES)
    # ===================================================================
    
    noise_level = config.get('noise_level', 'standard')
    kernel_time = 0
    
    if skip_quantum:
        print(f"\nLoading cached kernels (scales 1.0 & 3.0)...")
        kernels_train = {
            1.0: np.load(train_file_scale1),
            3.0: np.load(train_file_scale3)
        }
        kernels_test = {
            1.0: np.load(test_file_scale1),
            3.0: np.load(test_file_scale3)
        }
        print(f"  → Loaded from {kernel_dir}/")
    else:
        print(f"\nComputing quantum kernels at ZNE scales {ZNE_SCALES}...")
        
        # Get noise parameters
        noise_config = NOISE_CONFIGS.get(noise_level, NOISE_CONFIGS['standard'])
        print(f"Noise Model: 1q={noise_config['p_1q']*100:.2f}%, 2q={noise_config['p_2q']*100:.2f}%, readout={noise_config['p_ro']*100:.2f}%")
        
        kernels_train = {}
        kernels_test = {}
        
        feature_map = ZZFeatureMap(
            feature_dimension=k_features,
            reps=config['reps'],
            entanglement=config['entanglement']
        )
        
        print(f"Noisy quantum kernel configured (ZZFeatureMap, reps={config['reps']}, entanglement={config['entanglement']}, noise={noise_level})")
        
        print("\nComputing kernel matrices...")
        start_kernel = time.time()
        for scale in ZNE_SCALES:
            print(f"  → Computing kernels at noise scale {scale}...")
            _, backend, pm, _ = get_scaled_noise_model(scale_factor=scale, level=noise_level)
            sampler = AerSampler.from_backend(backend=backend, default_shots=config['shots'])
            fidelity = ComputeUncompute(sampler=sampler, pass_manager=pm)
            qkernel = FidelityQuantumKernel(fidelity=fidelity, feature_map=feature_map)
            
            kernels_train[scale] = qkernel.evaluate(x_vec=X_train_kbest)
            kernels_test[scale] = qkernel.evaluate(x_vec=X_test_kbest, y_vec=X_train_kbest)
        
        kernel_time = time.time() - start_kernel
        print(f"Kernel computation: {kernel_time:.2f}s")
        
        # Save kernels for future use
        np.save(train_file_scale1, kernels_train[1.0])
        np.save(test_file_scale1, kernels_test[1.0])
        np.save(train_file_scale3, kernels_train[3.0])
        np.save(test_file_scale3, kernels_test[3.0])
        print(f"Saved kernel matrices to {kernel_dir}/")
    
    # ===================================================================
    # 6. ERROR MITIGATION (ZNE + REM)
    # ===================================================================
    
    print(f"\nApplying Error Mitigation (ZNE + REM)...")
    
    # Get noise config for REM
    noise_config = NOISE_CONFIGS.get(noise_level, NOISE_CONFIGS['standard'])
    p_ro = noise_config['p_ro']
    n_qubits = k_features
    
    # Step 1: ZNE (Linear Richardson extrapolation)
    print(f"  → Step 1: Zero-Noise Extrapolation (Richardson)")
    kernel_train_zne = 1.5 * kernels_train[1.0] - 0.5 * kernels_train[3.0]
    kernel_test_zne = 1.5 * kernels_test[1.0] - 0.5 * kernels_test[3.0]
    
    # Step 2: REM (Readout error mitigation)
    print(f"  → Step 2: Readout Error Mitigation (p_ro={p_ro:.4f}, n_qubits={n_qubits})")
    matrix_train = apply_rem_to_kernel(kernel_train_zne, p_ro, n_qubits)
    matrix_test = apply_rem_to_kernel(kernel_test_zne, p_ro, n_qubits)
    
    # Ensure validity
    matrix_train = np.clip(matrix_train, 0, 1)
    matrix_test = np.clip(matrix_test, 0, 1)
    print(f"Error mitigation complete !")
    
    # ===================================================================
    # 7. TRAINING WITH FIXED C
    # ===================================================================
    
    print("\nUsing optimal C...")
    
    # Use fixed C=1.0 from classical RBF for all experiments
    best_c = 1.0
    print(f"  → Fixed C: {best_c} (from classical RBF)")
    
    # Train with fixed C
    start_train = time.time()
    best_model = SVC(kernel='precomputed', C=best_c, class_weight='balanced', random_state=42)
    best_model.fit(matrix_train, y_train)
    train_time = time.time() - start_train
    
    # For consistency, still calculate CV score
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    from sklearn.model_selection import cross_val_score
    cv_scores = cross_val_score(best_model, matrix_train, y_train, cv=cv, scoring='accuracy')
    cv_score = cv_scores.mean()
    
    print(f"  → CV Score: {cv_score:.4f}")
    print(f"  → Training time: {train_time:.2f}s")
    
    # ===================================================================
    # 8. EVALUATION
    # ===================================================================
    
    y_train_pred = best_model.predict(matrix_train)
    y_test_pred = best_model.predict(matrix_test)
    
    train_acc = accuracy_score(y_train, y_train_pred)
    test_acc = accuracy_score(y_test, y_test_pred)
    test_balanced_acc = balanced_accuracy_score(y_test, y_test_pred)
    spam_recall = recall_score(y_test, y_test_pred, pos_label=1)
    gen_gap = abs(train_acc - test_acc)
    
    print(f"  → Train Accuracy: {train_acc:.4f}")
    print(f"  → Test Accuracy: {test_acc:.4f}")
    print(f"  → Test Balanced Accuracy: {test_balanced_acc:.4f}")
    print(f"  → Spam Recall: {spam_recall:.4f}")
    print(f"  → Generalization Gap: {gen_gap:.4f}")
    
    # ===================================================================
    # 9. STORE RESULTS
    # ===================================================================
    
    all_results.append({
        'experiment_id': config['id'],
        'exp_number': exp_num,
        'samples': config['samples'],
        'k_features': k_features,
        'shots': config['shots'],
        'reps': config['reps'],
        'entanglement': config['entanglement'],
        'noise_level': noise_level,
        'selected_features': selected_features,
        'best_c': best_c,
        'cv_score': cv_score,
        'train_acc': train_acc,
        'test_acc': test_acc,
        'test_balanced_acc': test_balanced_acc,
        'spam_recall': spam_recall,
        'gen_gap': gen_gap,
        'kernel_time': kernel_time,
        'train_time': train_time,
        'y_test': y_test.tolist(),
        'y_pred': y_test_pred.tolist()
    })
    
    # Print classification report
    print("\nClassification Report:")
    print(classification_report(y_test, y_test_pred, zero_division=0))
    
    # Save mitigated kernel matrices
    np.save(f'{kernel_dir}/kernel_train_mitigated_{config["id"]}.npy', matrix_train)
    np.save(f'{kernel_dir}/kernel_test_mitigated_{config["id"]}.npy', matrix_test)
    print(f"Saved mitigated kernel matrices: kernel_train_mitigated_{config['id']}.npy, kernel_test_mitigated_{config['id']}.npy")

print("\n" + "=" * 80)
print("ALL EXPERIMENTS COMPLETE!")
print("=" * 80)

results_df = pd.DataFrame(all_results)
results_df.to_csv('em_qsvm_spambase_znerem_results.csv', index=False)
print(f"\nSaved {len(results_df)} experiments to: em_qsvm_spambase_znerem_results.csv")
print(f"Kernel matrices saved to: {kernel_dir}/")



EXPERIMENT 1/8: Baseline
  Samples: 300
  K Features: 3
  Shots: 1024
  Reps: 1
  Entanglement: linear
  Noise Level: standard



Dataset created: 300 train, 129 test
Data scaled successfully
Dropped 1 highly correlated features
SelectKBest: Selected 3 features:
     1. word_freq_remove
     2. char_freq_!
     3. char_freq_$

Loading cached kernels (scales 1.0 & 3.0)...
  → Loaded from kernels_em_znerem_spambase_final/

Applying Error Mitigation (ZNE + REM)...
  → Step 1: Zero-Noise Extrapolation (Richardson)
  → Step 2: Readout Error Mitigation (p_ro=0.0200, n_qubits=3)
Error mitigation complete !

Using optimal C...
  → Fixed C: 1.0 (from classical RBF)
  → CV Score: 0.8533
  → Training time: 0.03s
  → Train Accuracy: 0.8600
  → Test Accuracy: 0.8450
  → Test Balanced Accuracy: 0.8480
  → Spam Recall: 0.8627
  → Generalization Gap: 0.0150

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.83      0.87        78
           1       0.77      0.86      0.81        51

    accuracy                           0.84       129
   macro avg       0.84      0.85

In [ ]:
# ===================================================================
# CREATE DATAFRAME AND SAVE RESULTS
# ===================================================================

results_df = pd.DataFrame(all_results)
results_df.to_csv('qsvm_selectkbest_all_experiments.csv', index=False)

print("\n" + "=" * 80)
print("RESULTS SUMMARY")
print("=" * 80)
print(results_df[['experiment_id', 'samples', 'k_features', 'shots', 'reps', 'entanglement', 
                   'test_acc', 'test_balanced_acc', 'spam_recall', 'gen_gap', 'kernel_time']])
print(f"\nFull results saved to: qsvm_selectkbest_all_experiments.csv")

In [ ]:
# ===================================================================
# CONFUSION MATRICES FOR EACH EXPERIMENT
# ===================================================================
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import numpy as np

print("\nGenerating confusion matrices for all experiments...")

fig_rows = (len(experiments) + 3) // 4  # Calculate rows needed (4 plots per row)
fig, axes = plt.subplots(fig_rows, 4, figsize=(20, 5 * fig_rows))
axes = axes.flatten()  # Flatten to 1D array for easy indexing

for idx, result in enumerate(all_results):
    y_test = np.array(result['y_test'])
    y_pred = np.array(result['y_pred'])
    
    # Create confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Non-Spam', 'Spam'])
    
    # Plot on corresponding subplot
    disp.plot(ax=axes[idx], cmap='Blues', values_format='d')
    axes[idx].set_title(f"{result['experiment_id']}\nAcc: {result['test_acc']:.3f}", 
                        fontsize=10, fontweight='bold')
    axes[idx].grid(False)

# Hide unused subplots
for idx in range(len(all_results), len(axes)):
    axes[idx].axis('off')

plt.tight_layout()
plt.savefig('confusion_matrices_all_experiments.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Confusion matrices saved: confusion_matrices_all_experiments.png")


In [ ]:
# ===================================================================
# BEST CONFIGURATIONS ANALYSIS
# ===================================================================

print("\n" + "=" * 80)
print("BEST CONFIGURATIONS ANALYSIS")
print("=" * 80)


# ==========================================
# 1. BEST TEST ACCURACY
# ==========================================
best_acc_idx = results_df['test_acc'].idxmax()
best_acc = results_df.iloc[best_acc_idx]

print("\n" + "─" * 80)
print("1️BEST TEST ACCURACY (Primary Target)")
print("─" * 80)
print(f"\nExperiment ID: {best_acc['experiment_id']}")
print(f"  → Test Accuracy: {best_acc['test_acc']:.4f}")
print(f"  → Test Balanced Acc: {best_acc['test_balanced_acc']:.4f}")
print(f"  → Spam Recall: {best_acc['spam_recall']:.4f}")
print(f"  → Generalization Gap: {best_acc['gen_gap']:.4f}")
print(f"\nConfiguration:")
print(f"  → Samples: {int(best_acc['samples'])}")
print(f"  → Features (k): {int(best_acc['k_features'])}")
print(f"  → Shots: {int(best_acc['shots'])}")
print(f"  → Reps: {int(best_acc['reps'])}")
print(f"  → Entanglement: {best_acc['entanglement']}")
print(f"  → Optimal C: {best_acc['best_c']}")
print(f"\nTiming:")
print(f"  → Kernel Time: {best_acc['kernel_time']:.2f}s")
print(f"  → Training Time: {best_acc['train_time']:.2f}s")


# ==========================================
# 2. BEST SPAM RECALL
# ==========================================
best_recall_idx = results_df['spam_recall'].idxmax()
best_recall = results_df.iloc[best_recall_idx]

print("\n" + "─" * 80)
print("2️BEST SPAM RECALL (Spam Detection Focus)")
print("─" * 80)
print(f"\nExperiment ID: {best_recall['experiment_id']}")
print(f"  → Spam Recall: {best_recall['spam_recall']:.4f}")
print(f"  → Test Accuracy: {best_recall['test_acc']:.4f}")
print(f"  → Test Balanced Acc: {best_recall['test_balanced_acc']:.4f}")
print(f"  → Generalization Gap: {best_recall['gen_gap']:.4f}")
print(f"\nConfiguration:")
print(f"  → Samples: {int(best_recall['samples'])}")
print(f"  → Features (k): {int(best_recall['k_features'])}")
print(f"  → Shots: {int(best_recall['shots'])}")
print(f"  → Reps: {int(best_recall['reps'])}")
print(f"  → Entanglement: {best_recall['entanglement']}")
print(f"  → Optimal C: {best_recall['best_c']}")


# ==========================================
# 3. BEST GENERALIZATION
# ==========================================
best_gen_idx = results_df['gen_gap'].idxmin()
best_gen = results_df.iloc[best_gen_idx]

print("\n" + "─" * 80)
print("BEST GENERALIZATION (Lowest Overfitting)")
print("─" * 80)
print(f"\nExperiment ID: {best_gen['experiment_id']}")
print(f"  → Generalization Gap: {best_gen['gen_gap']:.4f}")
print(f"  → Test Accuracy: {best_gen['test_acc']:.4f}")
print(f"  → Spam Recall: {best_gen['spam_recall']:.4f}")
print(f"\nConfiguration:")
print(f"  → Samples: {int(best_gen['samples'])}")
print(f"  → Features (k): {int(best_gen['k_features'])}")
print(f"  → Shots: {int(best_gen['shots'])}")
print(f"  → Reps: {int(best_gen['reps'])}")
print(f"  → Entanglement: {best_gen['entanglement']}")


# ==========================================
# 4. BEST CROSS-VALIDATION SCORE
# ==========================================
best_cv_idx = results_df['cv_score'].idxmax()
best_cv = results_df.iloc[best_cv_idx]

print("\n" + "─" * 80)
print("BEST CROSS-VALIDATION SCORE (Training Reliability)")
print("─" * 80)
print(f"\nExperiment ID: {best_cv['experiment_id']}")
print(f"  → CV Score: {best_cv['cv_score']:.4f}")
print(f"  → Test Accuracy: {best_cv['test_acc']:.4f}")
print(f"  → Spam Recall: {best_cv['spam_recall']:.4f}")
print(f"  → Generalization Gap: {best_cv['gen_gap']:.4f}")
print(f"\nConfiguration:")
print(f"  → Samples: {int(best_cv['samples'])}")
print(f"  → Features (k): {int(best_cv['k_features'])}")
print(f"  → Shots: {int(best_cv['shots'])}")
print(f"  → Reps: {int(best_cv['reps'])}")
print(f"  → Entanglement: {best_cv['entanglement']}")


# ==========================================
# 5. FASTEST COMPUTATION
# ==========================================
fastest_idx = results_df['kernel_time'].idxmin()
fastest = results_df.iloc[fastest_idx]

print("\n" + "─" * 80)
print("FASTEST COMPUTATION (Efficiency)")
print("─" * 80)
print(f"\nExperiment ID: {fastest['experiment_id']}")
print(f"  → Kernel Time: {fastest['kernel_time']:.2f}s")
print(f"  → Test Accuracy: {fastest['test_acc']:.4f}")
print(f"  → Spam Recall: {fastest['spam_recall']:.4f}")
print(f"\nConfiguration:")
print(f"  → Samples: {int(fastest['samples'])}")
print(f"  → Features (k): {int(fastest['k_features'])}")
print(f"  → Shots: {int(fastest['shots'])}")
print(f"  → Reps: {int(fastest['reps'])}")
print(f"  → Entanglement: {fastest['entanglement']}")


# ==========================================
# COMPARISON TABLE
# ==========================================
print("\n" + "=" * 80)
print("COMPARISON OF BEST CONFIGURATIONS")
print("=" * 80)

comparison_data = {
    'Category': ['Best Accuracy', 'Best Recall', 'Best Generalization', 'Best CV', 'Fastest'],
    'Exp ID': [
        best_acc['experiment_id'],
        best_recall['experiment_id'],
        best_gen['experiment_id'],
        best_cv['experiment_id'],
        fastest['experiment_id']
    ],
    'Test Acc': [
        f"{best_acc['test_acc']:.4f}",
        f"{best_recall['test_acc']:.4f}",
        f"{best_gen['test_acc']:.4f}",
        f"{best_cv['test_acc']:.4f}",
        f"{fastest['test_acc']:.4f}"
    ],
    'Spam Recall': [
        f"{best_acc['spam_recall']:.4f}",
        f"{best_recall['spam_recall']:.4f}",
        f"{best_gen['spam_recall']:.4f}",
        f"{best_cv['spam_recall']:.4f}",
        f"{fastest['spam_recall']:.4f}"
    ],
    'Gen Gap': [
        f"{best_acc['gen_gap']:.4f}",
        f"{best_recall['gen_gap']:.4f}",
        f"{best_gen['gen_gap']:.4f}",
        f"{best_cv['gen_gap']:.4f}",
        f"{fastest['gen_gap']:.4f}"
    ],
    'Shots': [
        int(best_acc['shots']),
        int(best_recall['shots']),
        int(best_gen['shots']),
        int(best_cv['shots']),
        int(fastest['shots'])
    ],
    'Reps': [
        int(best_acc['reps']),
        int(best_recall['reps']),
        int(best_gen['reps']),
        int(best_cv['reps']),
        int(fastest['reps'])
    ],
    'Entangle': [
        best_acc['entanglement'],
        best_recall['entanglement'],
        best_gen['entanglement'],
        best_cv['entanglement'],
        fastest['entanglement']
    ]
}

comparison_df = pd.DataFrame(comparison_data)
print("\n" + comparison_df.to_string(index=False))


# ==========================================
# EXPORT CONFIGURATION
# ==========================================
print("\n" + "=" * 80)
print("CONFIGURATION EXPORT")
print("=" * 80)

qsvm_config = {
    'best_accuracy': {
        'experiment_id': best_acc['experiment_id'],
        'test_acc': float(best_acc['test_acc']),
        'spam_recall': float(best_acc['spam_recall']),
        'samples': int(best_acc['samples']),
        'k_features': int(best_acc['k_features']),
        'shots': int(best_acc['shots']),
        'reps': int(best_acc['reps']),
        'entanglement': best_acc['entanglement'],
        'optimal_C': float(best_acc['best_c']),
        'kernel_time': float(best_acc['kernel_time'])
    },
    'best_recall': {
        'experiment_id': best_recall['experiment_id'],
        'spam_recall': float(best_recall['spam_recall']),
        'test_acc': float(best_recall['test_acc']),
        'samples': int(best_recall['samples']),
        'k_features': int(best_recall['k_features']),
        'shots': int(best_recall['shots']),
        'reps': int(best_recall['reps']),
        'entanglement': best_recall['entanglement']
    },
    'best_generalization': {
        'experiment_id': best_gen['experiment_id'],
        'gen_gap': float(best_gen['gen_gap']),
        'test_acc': float(best_gen['test_acc']),
        'samples': int(best_gen['samples']),
        'k_features': int(best_gen['k_features'])
    },
    'fastest_computation': {
        'experiment_id': fastest['experiment_id'],
        'kernel_time': float(fastest['kernel_time']),
        'test_acc': float(fastest['test_acc']),
        'shots': int(fastest['shots']),
        'reps': int(fastest['reps'])
    }
}

# Save configuration
import json
with open('ideal_qsvm_best_configurations.json', 'w') as f:
    json.dump(qsvm_config, f, indent=4)

print("\n✓ Configuration saved to: ideal_qsvm_best_configurations.json")
print("=" * 80)


In [ ]:
# ===================================================================
# VISUALIZATION
# ===================================================================

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# ==========================================
# CONFIGURATION & STYLE
# ==========================================
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (18, 5)
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.titleweight'] = 'bold'
plt.rcParams['lines.linewidth'] = 2
plt.rcParams['lines.markersize'] = 8

def plot_qsvm_experiments(df):
    """
    Generates standardized visualizations for EM-QSVM experiments.
    Handles: Baseline, Exp1 (shots), Exp2 (reps), Exp3 (entanglement), Exp4 (noise_level)
    """
    
    # Map Experiment SERIES to parameters
    exp_map = {
        'Exp1': {
            'param': 'shots',
            'xlabel': 'Number of Shots (Measurements)',
            'log_x': True,
            'type': 'line',
            'filter': 'Exp1'
        },
        'Exp2': {
            'param': 'reps',
            'xlabel': 'Circuit Depth (Repetitions)',
            'log_x': False,
            'type': 'line',
            'filter': 'Exp2'
        },
        'Exp3': {
            'param': 'entanglement',
            'xlabel': 'Entanglement Topology',
            'log_x': False,
            'type': 'bar',
            'filter': 'Exp3'
        },
        'Exp4': {
            'param': 'noise_level',
            'xlabel': 'Noise Level',
            'log_x': False,
            'type': 'bar',
            'filter': 'Exp4'
        }
    }
    
    # Iterate through each experiment series
    for group_key, config in exp_map.items():
        # Filter data for this series
        if group_key in ['Exp2', 'Exp3']:
            # Include Baseline as reference for Exp2 and Exp3
            subset = df[df['experiment_id'].str.contains(config['filter'])].copy()
            baseline = df[df['experiment_id'] == 'Baseline'].copy()
            if not baseline.empty and config['param'] in ['reps', 'entanglement']:
                subset = pd.concat([baseline, subset], ignore_index=True)
        elif group_key == 'Exp4':
            # For Exp4, include Baseline as standard noise
            subset = df[df['experiment_id'].str.contains(config['filter'])].copy()
            baseline = df[df['experiment_id'] == 'Baseline'].copy()
            if not baseline.empty:
                subset = pd.concat([baseline, subset], ignore_index=True)
        else:
            # Exp1 includes Baseline automatically (1024 shots)
            subset = df[df['experiment_id'].str.contains(config['filter'])].copy()
            baseline = df[df['experiment_id'] == 'Baseline'].copy()
            if not baseline.empty and config['param'] == 'shots':
                subset = pd.concat([baseline, subset], ignore_index=True)
        
        if subset.empty:
            print(f"No data found for {group_key}")
            continue
            
        param = config['param']
        
        # Sort by parameter for line plots
        if config['type'] == 'line':
            subset = subset.sort_values(by=param)
            
        print(f"\n{'─'*80}")
        print(f"Plotting {group_key}: Effect of {config['xlabel']}")
        print(f"{'─'*80}")
        print(f"Found {len(subset)} experiments in this series")
        
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        
        # -------------------------------------------------------
        # SUBPLOT 1: PERFORMANCE (Accuracy vs Recall)
        # -------------------------------------------------------
        if config['type'] == 'bar':
            x_pos = np.arange(len(subset))
            width = 0.35
            axes[0].bar(x_pos - width/2, subset['test_acc'], width, 
                       label='Test Accuracy', color='#1f77b4', alpha=0.9, edgecolor='black')
            axes[0].bar(x_pos + width/2, subset['spam_recall'], width, 
                       label='Spam Recall', color='#ff7f0e', alpha=0.9, edgecolor='black')
            axes[0].set_xticks(x_pos)
            axes[0].set_xticklabels(subset[param], rotation=0)
        else:
            axes[0].plot(subset[param], subset['test_acc'], 'o-', 
                        label='Test Accuracy', color='#1f77b4', linewidth=2.5, markersize=10)
            axes[0].plot(subset[param], subset['spam_recall'], 's--', 
                        label='Spam Recall', color='#ff7f0e', linewidth=2.5, markersize=10)
            if config['log_x']: 
                axes[0].set_xscale('log', base=2)
                axes[0].set_xticks(subset[param].unique())
                axes[0].set_xticklabels(subset[param].unique())

        axes[0].set_xlabel(config['xlabel'], fontsize=13, fontweight='bold')
        axes[0].set_ylabel('Score', fontsize=13, fontweight='bold')
        axes[0].set_title(f'Performance vs {param.replace("_", " ").title()}', 
                         fontsize=14, fontweight='bold', pad=15)
        axes[0].set_ylim(0, 1.05)
        axes[0].legend(loc='lower right', fontsize=11)
        axes[0].grid(True, alpha=0.3, linestyle='--')

        # -------------------------------------------------------
        # SUBPLOT 2: GENERALIZATION GAP
        # -------------------------------------------------------
        if config['type'] == 'bar':
            bars = axes[1].bar(subset[param], subset['gen_gap'], 
                              color='#d62728', alpha=0.7, edgecolor='black')
            axes[1].set_xticks(range(len(subset)))
            axes[1].set_xticklabels(subset[param], rotation=0)
            
            # Add value labels on bars
            for bar in bars:
                height = bar.get_height()
                axes[1].text(bar.get_x() + bar.get_width()/2., height,
                           f'{height:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
        else:
            axes[1].plot(subset[param], subset['gen_gap'], 'D-', 
                        color='#d62728', linewidth=2.5, markersize=10)
            if config['log_x']: 
                axes[1].set_xscale('log', base=2)
                axes[1].set_xticks(subset[param].unique())
                axes[1].set_xticklabels(subset[param].unique())
            
        axes[1].set_xlabel(config['xlabel'], fontsize=13, fontweight='bold')
        axes[1].set_ylabel('Gap (Train Acc - Test Acc)', fontsize=13, fontweight='bold')
        axes[1].set_title('Generalization Gap (Lower = Less Overfitting)', 
                         fontsize=14, fontweight='bold', pad=15)
        axes[1].grid(True, alpha=0.3, linestyle='--')
        axes[1].axhline(y=0, color='black', linestyle='-', linewidth=0.8, alpha=0.5)

        # -------------------------------------------------------
        # SUBPLOT 3: COMPUTATIONAL COST
        # -------------------------------------------------------
        if config['type'] == 'bar':
            bars = axes[2].bar(subset[param], subset['kernel_time'], 
                              color='#2ca02c', alpha=0.7, edgecolor='black')
            axes[2].set_xticks(range(len(subset)))
            axes[2].set_xticklabels(subset[param], rotation=0)
            
            # Add time labels
            for bar in bars:
                height = bar.get_height()
                axes[2].text(bar.get_x() + bar.get_width()/2., height,
                           f'{height:.1f}s', ha='center', va='bottom', fontsize=10, fontweight='bold')
        else:
            axes[2].plot(subset[param], subset['kernel_time'], '^-', 
                        color='#2ca02c', linewidth=2.5, markersize=10)
            if config['log_x']: 
                axes[2].set_xscale('log', base=2)
                axes[2].set_xticks(subset[param].unique())
                axes[2].set_xticklabels(subset[param].unique())

        axes[2].set_xlabel(config['xlabel'], fontsize=13, fontweight='bold')
        axes[2].set_ylabel('Kernel Computation Time (s)', fontsize=13, fontweight='bold')
        axes[2].set_title('Computational Efficiency', fontsize=14, fontweight='bold', pad=15)
        axes[2].grid(True, alpha=0.3, linestyle='--')
        
        plt.tight_layout()
        plt.savefig(f'em_qsvm_{group_key}_analysis.png', dpi=300, bbox_inches='tight')
        plt.show()
        print(f"✓ Saved: em_qsvm_{group_key}_analysis.png")

    # -------------------------------------------------------
    # SUMMARY HEATMAP
    # -------------------------------------------------------
    print(f"\n{'='*80}")
    print("GLOBAL PERFORMANCE HEATMAP")
    print(f"{'='*80}")
    
    plt.figure(figsize=(12, 10))
    
    # Metrics to display
    metrics = ['test_acc', 'test_balanced_acc', 'spam_recall', 'gen_gap', 'cv_score']
    
    # Prepare data
    heatmap_data = df.set_index('experiment_id')[metrics].T
    
    # Plot
    sns.heatmap(heatmap_data, annot=True, fmt='.3f', cmap='RdYlGn', 
                vmin=0, vmax=1, linewidths=1, linecolor='white',
                cbar_kws={'label': 'Score'}, 
                annot_kws={'fontsize': 10, 'fontweight': 'bold'})
    
    plt.title('EM-QSVM Performance Metrics Across All Experiments', 
             fontsize=16, fontweight='bold', pad=20)
    plt.xlabel('Experiment ID', fontsize=13, fontweight='bold')
    plt.ylabel('Metrics', fontsize=13, fontweight='bold')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig('em_qsvm_heatmap_summary.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("✓ Saved: em_qsvm_heatmap_summary.png")

try:
    # Load results if not already in memory
    if 'results_df' not in locals():
        results_df = pd.read_csv('em_qsvm_spambase_znerem_results.csv')
    
    print("="*80)
    print("ERROR-MITIGATED QUANTUM SVM VISUALIZATION SUITE")
    print("="*80)
    print(f"\nLoaded {len(results_df)} experiments")
    print(f"Columns: {list(results_df.columns)}")
    
    # Run visualizations
    plot_qsvm_experiments(results_df)
    
    print("\n" + "="*80)
    print("ALL VISUALIZATIONS COMPLETE")
    print("="*80)
    print("\nGenerated Files:")
    print("  1. em_qsvm_Exp1_analysis.png (Shot Noise Analysis)")
    print("  2. em_qsvm_Exp2_analysis.png (Circuit Depth Analysis)")
    print("  3. em_qsvm_Exp3_analysis.png (Entanglement Comparison)")
    print("  4. em_qsvm_Exp4_analysis.png (Noise Level Analysis)")
    print("  5. em_qsvm_heatmap_summary.png (Global Performance Heatmap)")

except FileNotFoundError:
    print("Error: 'em_qsvm_spambase_znerem_results.csv' not found")
    print("   Make sure you've run the main experiment loop first!")
except Exception as e:
    print(f"Error: {e}")
    import traceback
    traceback.print_exc()
